# NKC cohort 2003–2017

Exploratory look at the SCKN-labelable slice of the NKC translation dataset.

**Why 2003–2017?**
- 2003 — SCKN charts begin; earlier records can never carry a positive label.
- 2017 — natural candidate for the train/test temporal cutoff in M5. Books published 2018+ form the held-out test set (the model has never "seen" the market conditions that led to their translation).

This notebook answers: how big is the cohort, how clean is the data, and what matching-key coverage can we expect going into M4?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CSV_PATH = REPO / "data" / "interim" / "nkc_translations.csv"

raw = pd.read_csv(CSV_PATH, dtype=str).fillna("")
raw["year"] = pd.to_numeric(raw["czech_pub_year"], errors="coerce")

df = raw[raw["year"].between(2003, 2017)].copy()
print(f"Full CSV:       {len(raw):>7,} rows")
print(f"2003–2017 slice:{len(df):>7,} rows ({len(df)/len(raw):.1%} of total)")

## Volume by year

How many translations were published in Czech each year? Stable volume is good — it means the training signal is spread evenly across the timeline rather than dominated by a single boom year.

In [ ]:
by_year = df.groupby("year").size().rename("translations")

ax = by_year.plot.bar(figsize=(10, 3), color="steelblue")
ax.set_xlabel("Czech publication year")
ax.set_ylabel("translations")
ax.set_title("Czech translations per year (2003–2017)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(by_year.to_string())
print(f"\nTotal: {by_year.sum():,}  |  Mean/year: {by_year.mean():.0f}  |  Min: {by_year.min():,}  Max: {by_year.max():,}")

## Matching-key coverage

These four fields are the join keys used in M4:
- **ISBN** → SCKN → NKC exact match (primary)
- **OCLC** → NKC → Goodreads exact match (primary)
- **original_title** → NKC → Goodreads fuzzy fallback when OCLC absent
- **author** → required by all fuzzy-match strategies

In [ ]:
def pct(col): return (df[col] != "").mean()

coverage = pd.Series({
    "czech_isbn (020 $a)": pct("czech_isbn"),
    "oclc (035 $a)": pct("oclc"),
    "original_title (240 $a)": pct("original_title"),
    "author (100 $a)": pct("author"),
    "secondary_authors (700 $a)": pct("secondary_authors"),
    "genres (655 $a)": pct("genres"),
})

coverage.sort_values().plot.barh(figsize=(7, 3.5))
plt.xlim(0, 1)
plt.xlabel("coverage on 2003–2017 subset")
plt.title("Field coverage — matching keys")
plt.tight_layout()
plt.show()

coverage.sort_values(ascending=False).map("{:.2%}".format).to_frame("coverage")

## Coverage drift over the cohort

Does coverage change across the 2003–2017 window? A dip in one year could indicate a cataloguing batch that was processed differently.

In [ ]:
trend = df.groupby("year").agg(
    isbn=("czech_isbn", lambda s: (s != "").mean()),
    oclc=("oclc", lambda s: (s != "").mean()),
    orig_title=("original_title", lambda s: (s != "").mean()),
    author=("author", lambda s: (s != "").mean()),
)

trend.plot(figsize=(10, 4), marker="o", markersize=3)
plt.ylim(0, 1.05)
plt.ylabel("coverage")
plt.title("Matching-key coverage by year (2003–2017)")
plt.tight_layout()
plt.show()

## Source language breakdown

What languages are Czech publishers translating from? English dominance is expected; the long tail matters for genre coverage.

In [ ]:
# Each row can have pipe-joined languages (e.g. "eng|ger") — explode to count each
langs = (
    df["source_lang"]
    .str.split("|")
    .explode()
    .loc[lambda s: s != ""]
    .value_counts()
)

langs.head(15).sort_values().plot.barh(figsize=(7, 5))
plt.xlabel("records")
plt.title("Top 15 source languages (2003–2017)")
plt.tight_layout()
plt.show()

pct_eng = langs.get("eng", 0) / langs.sum()
print(f"English share: {pct_eng:.1%}")
langs.head(15).to_frame("records")

## Records missing ISBN

ISBN is the primary SCKN→NKC join key. Records without it will fall back to fuzzy title+author matching in M4 — which is more expensive and error-prone. Understanding what's missing helps scope the fallback work.

In [ ]:
no_isbn = df[df["czech_isbn"] == ""]
print(f"Records without ISBN: {len(no_isbn):,} ({len(no_isbn)/len(df):.2%})")
print()

# Are they concentrated in certain years?
missing_by_year = no_isbn.groupby("year").size()
total_by_year = df.groupby("year").size()
(missing_by_year / total_by_year).plot.bar(figsize=(10, 3), color="salmon")
plt.ylabel("fraction without ISBN")
plt.title("Missing-ISBN rate by year")
plt.tight_layout()
plt.show()

# Sample of no-ISBN records
no_isbn[["nkc_id", "oclc", "czech_title", "author", "genres"]].head(10)

## Author format

NKC stores authors as **"Surname, Given"** (e.g. `King, Stephen`) while SCKN uses **"Given Surname"** (e.g. `Stephen King`). M4 fuzzy matching must normalise both to a canonical form before comparing.

Some NKC author entries also carry a trailing comma from ISBD punctuation rules — the parser strips it, but let's verify and also look at format diversity.

In [ ]:
# Show a sample of author values to illustrate the format
sample_authors = (
    df[df["author"] != ""]
    .drop_duplicates("author")
    .sample(20, random_state=42)["author"]
    .sort_values()
    .tolist()
)
for a in sample_authors:
    print(repr(a))

# Any authors that still end with a comma (parser should have stripped these)
trailing_comma = df[df["author"].str.endswith(",")]
print(f"\nAuthors still ending with comma: {len(trailing_comma)}")

## Sample rows

End-to-end sanity check — do all fields look sensible together?

In [ ]:
# Show rows that have all four matching keys populated — the easiest M4 cases
complete = df[
    (df["czech_isbn"] != "") &
    (df["oclc"] != "") &
    (df["original_title"] != "") &
    (df["author"] != "")
]
pct_complete = len(complete) / len(df)
print(f"Records with all four keys: {len(complete):,} ({pct_complete:.1%})")
complete[["nkc_id", "czech_isbn", "oclc", "czech_title", "original_title", "author", "source_lang", "year"]].head(10)